In [ ]:
%run langchain_common_notebook.py

In [ ]:
docs = [
    "Hi there!",
    "Oh, hello!",
    "What's your name?",
    "My friends call me World",
    "Hello World! What is going on? Where have you been? I haven't seen you in a while. How have you been? I hope everything is going well for you. It's been a long time since we last spoke. Let's catch up soon and talk about what's new in our lives."
]

embeddings = databricks_embeddings.embed_documents(docs)

In [ ]:
# This is just to show the embeddings in a nice table format. 
import pandas as pd
pd.DataFrame(embeddings)


### Text String Embedding (PGVector.from_texts)

In [ ]:
# from_texts: loads text strings into the vector store, embedding them in the process. 
# This is the most common way to load data into a vector store.    
db = PGVector.from_texts(docs, 
                         databricks_embeddings, 
                         collection_name="test_sentences", 
                         connection=pgvectordb_conn,
                         use_jsonb=True)

In [ ]:
# PGVector returns distances here: lower values indicate closer vectors.
# You can also get similarity search without scores using `db.similarity_search()`, which returns just the documents.
scored_results = db.similarity_search_with_score("Hi there!", k=3)

for i, (doc, score) in enumerate(scored_results, 1):
    print(f"{i}. distance={score:.6f} | text={doc.page_content}")

In [ ]:
# Retriever interface (RAG-friendly)
retriever = db.as_retriever(search_kwargs={"k": 3})
retrieved_docs = retriever.invoke("Hi there")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"{i}. {doc.page_content}")


### Document Embeddings (PGVector.from_documents)

In [ ]:
from langchain_community.document_loaders import TextLoader

raw_documents = TextLoader('./llm_notes.txt', encoding="utf-8").load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.split_documents(raw_documents)

In [ ]:
db = PGVector.from_documents(   documents,
                                databricks_embeddings,
                                collection_name="llm_notes",
                                connection=pgvectordb_conn,
                                use_jsonb=True,
                                pre_delete_collection=True)


In [ ]:
enable_logging()
db.similarity_search("softmax", k=4)
disable_logging()

### Add New Documents

In [ ]:
import uuid
from langchain_core.documents import Document

# add_documents: adds documents to the vector store, embedding them in the process.
# This is useful for adding new documents to an existing collection. 
# You can also specify your own document IDs, as shown below.
enable_logging()

ids = [str(uuid.uuid4()), str(uuid.uuid4())]
db.add_documents(
    [
        Document(
            page_content="there are cats in the pond",
            metadata={"location": "pond", "topic": "animals"},
        ),
        Document(
            page_content="ducks are also found in the pond",
            metadata={"location": "pond", "topic": "animals"},
        ),
    ],
    ids=ids,
)

disable_logging()
